# การสร้างแอปพลิเคชันสร้างภาพ (OpenAI)

มีอะไรมากกว่าการสร้างข้อความด้วย LLMs คุณยังสามารถสร้างภาพจากคำอธิบายข้อความได้ ภาพในฐานะสื่อมีประโยชน์ในหลายสาขาเช่น MedTech, สถาปัตย์, การท่องเที่ยว, การพัฒนาเกม, การตลาด และอื่น ๆ ในบทเรียนนี้เราจะทำงานกับโมเดล **GPT Image** ในแพลตฟอร์ม OpenAI ปัจจุบัน

## เป้าหมายการเรียนรู้

เมื่อจบบทเรียนนี้คุณจะสามารถ:

- อธิบายว่าการสร้างภาพคืออะไรและมีประโยชน์ในด้านใดบ้าง
- เข้าใจตระกูลโมเดล `gpt-image` และความแตกต่างจากโมเดล DALL·E รุ่นเก่า
- สร้างแอปพลิเคชันสร้างภาพและแก้ไขภาพได้

## การสร้างภาพคืออะไร?

โมเดลสร้างภาพสร้างภาพจากข้อความคำสั่งโมเดลสมัยใหม่เช่น `gpt-image` จะเรียนรู้ความสัมพันธ์ระหว่างข้อความและภาพในระหว่างการฝึก จากนั้นจะเปลี่ยนเสียงรบกวนสุ่มเป็นภาพที่ตรงกับคำอธิบายของคุณตามลำดับ

ตระกูลโมเดลภาพที่มีชื่อเสียงสองตระกูลคือ:

- **`gpt-image` (OpenAI)** — เจเนอเรชันปัจจุบันที่ใช้ในบทเรียนนี้ รองรับการสร้างภาพจากข้อความและการแก้ไขภาพ (วาดเติมส่วนที่ขาดด้วยหน้ากาก)
- **Midjourney** — โมเดลของบุคคลที่สามยอดนิยมที่มีบริการและเวิร์กโฟลว์บน Discord เป็นของตัวเอง

> โมเดลรุ่นเก่าของ OpenAI — **DALL·E 2** และ **DALL·E 3** — เป็นรุ่นเก่า DALL·E 3 ไม่สามารถใช้งานสำหรับการปรับใช้ใหม่ และฟีเจอร์เช่น `create_variation` มีใน DALL·E 2 เท่านั้น ใช้โมเดล `gpt-image` สำหรับแอปใหม่

> **สำคัญ:** โมเดล `gpt-image` จะส่งคืนภาพที่สร้างเป็น **base64** (`b64_json`) ไม่ใช่ URL โค้ดของคุณจะถอดรหัสสตริง base64 เป็นไบต์และบันทึก — ไม่มี URL ภาพให้ดาวน์โหลด


## การสร้างแอปพลิเคชันสร้างภาพแรกของคุณ

สิ่งที่จำเป็นสำหรับการสร้างแอปพลิเคชันสร้างภาพคืออะไร? คุณต้องใช้ไลบรารีต่อไปนี้:

- **python-dotenv**, แนะนำอย่างยิ่งให้ใช้ไลบรารีนี้เพื่อเก็บความลับของคุณไว้ในไฟล์ *.env* แยกจากโค้ด
- **openai**, ไลบรารีนี้คือสิ่งที่คุณจะใช้ในการติดต่อกับ OpenAI API
- **pillow**, สำหรับทำงานกับภาพในภาษา Python


1. สร้างไฟล์ *.env* โดยมีเนื้อหาดังนี้:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. รวบรวมไลบรารีข้างต้นในไฟล์ที่ชื่อ *requirements.txt* ดังนี้:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. ต่อไป สร้างสภาพแวดล้อมเสมือนและติดตั้งไลบรารี:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> สำหรับ Windows ให้ใช้คำสั่งต่อไปนี้เพื่อสร้างและเปิดใช้งานสภาพแวดล้อมเสมือนของคุณ:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. เพิ่มโค้ดต่อไปนี้ในไฟล์ที่ชื่อ *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # นำเข้า dotenv
    dotenv.load_dotenv()

    # สร้างวัตถุ OpenAI (อ่าน OPENAI_API_KEY จากไฟล์ .env ของคุณ)
    client = openai.OpenAI()


    try:
        # สร้างภาพโดยใช้ API สร้างภาพ
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ป้อนข้อความ prompt ของคุณที่นี่
            size='1024x1024',
            n=1
        )
        # ตั้งค่าที่เก็บรูปภาพ
        image_dir = os.path.join(os.curdir, 'images')

        # หากไดเรกทอรีไม่มีอยู่ ให้สร้างขึ้น
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # เริ่มต้นเส้นทางภาพ (โปรดสังเกตชนิดไฟล์ควรเป็น png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # โมเดล gpt-image จะส่งคืนภาพในรูปแบบ base64 (b64_json) ไม่ใช่ URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # แสดงภาพในโปรแกรมดูภาพเริ่มต้น
        image = Image.open(image_path)
        image.show()

    # จับข้อยกเว้น
    except openai.BadRequestError as err:
        print(err)

    ```

มาอธิบายโค้ดนี้กัน:

- ก่อนอื่น เรานำเข้าไลบรารีที่เราต้องการ รวมถึงไลบรารี OpenAI ไลบรารี dotenv โมดูล base64 และไลบรารี Pillow

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- หลังจากนั้น เราสร้างไคลเอนต์ ซึ่งจะอ่านคีย์ API จากไฟล์ ``.env`` ของคุณ

    ```python
    # สร้างวัตถุ OpenAI
    client = openai.OpenAI()
    ```

- ถัดมา เราสร้างภาพ:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ป้อนข้อความพร้อมท์ของคุณที่นี่
        size='1024x1024',
        n=1
    )
    ```

    โมเดล `gpt-image` จะคืนค่าภาพในรูปแบบสตริง **base64** ใน `data[0].b64_json` เราทำการถอดรหัสเป็นไบต์และบันทึกลงในไฟล์ — ไม่มี URL สำหรับดาวน์โหลด

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- สุดท้าย เราเปิดภาพและใช้โปรแกรมดูภาพมาตรฐานเพื่อแสดงภาพนั้น:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### รายละเอียดเพิ่มเติมเกี่ยวกับการสร้างภาพ

มาดูพารามิเตอร์ของ `images.generate` กัน:

- **model** คือโมเดลภาพที่ใช้ (เช่น `gpt-image-1`)
- **prompt** คือข้อความคำสั่งเพื่อสร้างภาพ ที่นี่คือ "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils"
- **size** คือขนาดของภาพที่สร้าง (เช่น `1024x1024`, `1536x1024`, `1024x1536` หรือ `"auto"`)
- **n** คือจำนวนภาพที่สร้าง ที่นี่เราสร้างหนึ่งภาพ

> โมเดลภาพจะไม่มีพารามิเตอร์ `temperature` — นั่นคือการควบคุมการสร้างข้อความ ในการได้ความหลากหลาย ให้เรียก API อีกครั้ง; หากต้องการลดความหลากหลาย ให้ทำคำสั่งของคุณให้เฉพาะเจาะจงขึ้น

## ความสามารถเพิ่มเติมของการสร้างภาพ

คุณเห็นแล้วว่าการสร้างภาพด้วย Python ไม่กี่บรรทัดง่ายเพียงใด โมเดล `gpt-image` ยังสามารถ **แก้ไข** ภาพที่มีอยู่แล้วได้ โดยการระบุภาพ หน้ากาก **mask** (ซึ่งเป็นตัวระบุพื้นที่ที่จะเปลี่ยนแปลง) และคำสั่ง คุณสามารถเปลี่ยนแปลงส่วนหนึ่งของภาพ — เช่น เพิ่มหมวกให้กระต่ายของเรา

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# การแก้ไขจะถูกส่งกลับในรูปแบบ base64 ด้วย
edited_image = base64.b64decode(response.data[0].b64_json)
```

ภาพต้นฉบับมีเพียงกระต่ายเท่านั้น ภาพสุดท้ายมีหมวกบนกระต่าย


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
